# LAB-D4-01: Accuracy Is Not Enough

**Purpose:** Choose metrics and operating thresholds from asymmetric error consequences, then expose how a high aggregate score can hide complete rare-class failure.

**Objectives:** `OBJ-D4-01`  
**Estimated duration:** 40 minutes live; completed CPU path under 30 seconds  
**Prerequisites:** [LESSON-D4-01](../student-guide/day-4-student-guide.md#lesson-d4-01---evaluation-begins-with-consequences), [ACT-D4-01](../challenges/day-4-challenges.md#act-d4-01---cost-council), `OBJ-D2-02`, and Day 3 split validity  
**Environment:** CPU; NumPy, matplotlib, scikit-learn; generated local data; no network or files

Workflow: **Name consequences -> Predict -> Compare baselines -> Implement metrics -> Sweep threshold -> Inspect synchronized evidence -> Choose -> Explain -> Extend**. Fraud is class `1`. The cost tables are teaching scenarios, not complete stakeholder-impact models.

In [ ]:
import platform
import time

import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, confusion_matrix, precision_recall_curve,
    precision_recall_fscore_support, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 4101
MODEL_SEED = 4102
DEFAULT_THRESHOLD = 0.50
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.2})
print(f"Python {platform.python_version()} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}")
print("Required path: CPU, local generated data, no network. Runtime target: under 30 seconds.")

## Recap: Metrics Serve a Decision

Accuracy asks how often a class decision matches. Precision asks what fraction of predicted fraud is fraud. Recall asks what fraction of fraud is found. F1 balances precision and recall, but it does not encode a scenario's actual costs. AUC summarizes ranking over many thresholds; it does not select the operating threshold.

In [ ]:
started = time.perf_counter()
X, y = make_classification(
    n_samples=6000, n_features=10, n_informative=5, n_redundant=2,
    weights=[0.96, 0.04], class_sep=1.20, flip_y=0.01, random_state=SEED,
)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED,
)
assert 0.95 <= np.mean(y_val == 0) <= 0.97
print(f"Train/validation: {X_train.shape}/{X_val.shape}; validation fraud rate: {y_val.mean():.3f}")

## Predict Before Evidence

Commit before calculating anything: predict majority-baseline accuracy and fraud recall, name the costlier error in each scenario, choose a metric set, and predict how lowering the threshold changes false negatives and false positives.

In [ ]:
metric_predictions = {
    "majority_accuracy_band": "",
    "majority_fraud_recall": "",
    "scenario_one_costlier_error": "",
    "scenario_two_costlier_error": "",
    "metric_set_and_why": "",
    "lower_threshold_effect": "",
}
assert all(value.strip() for value in metric_predictions.values()), (
    "Prediction gate: complete every field before revealing baseline metrics."
)

## Focused TODO: One Metric Row

Implement `metric_row`. Use confusion-matrix orientation `[[TN, FP], [FN, TP]]`, set `labels=[0,1]`, and pass `zero_division=0` so a no-positive-prediction baseline is informative rather than noisy.

In [ ]:
def metric_row(y_true, predictions):
    # TODO: return TN, FP, FN, TP, accuracy, precision, recall, and F1.
    raise NotImplementedError("TODO: implement the zero_division-safe metric row")

In [ ]:
majority_predictions = np.zeros_like(y_val)
majority_metrics = metric_row(y_val, majority_predictions)
print("Majority baseline:", majority_metrics)
assert 0.95 <= majority_metrics["accuracy"] <= 0.97
assert majority_metrics["recall"] == 0.0 and majority_metrics["f1"] == 0.0

## Observe the Accuracy-Only Failure

An accuracy-only selector may approve the majority baseline. State why that rule fails before fitting a probability model. Repair the *evaluation rule*: do not assume changing the classifier is always the first action.

In [ ]:
accuracy_only_diagnosis = {"why_it_fails": "", "replacement_evidence": ""}
assert all(value.strip() for value in accuracy_only_diagnosis.values())

In [ ]:
probability_model = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=MODEL_SEED)),
])
probability_model.fit(X_train, y_train)
fraud_probability = probability_model.predict_proba(X_val)[:, 1]
default_predictions = (fraud_probability >= DEFAULT_THRESHOLD).astype(int)
default_metrics = metric_row(y_val, default_predictions)
default_metrics["roc_auc"] = float(roc_auc_score(y_val, fraud_probability))
default_metrics["average_precision"] = float(average_precision_score(y_val, fraud_probability))
print("Probability model at threshold 0.50:", default_metrics)
assert 0.55 <= default_metrics["recall"] <= 0.85

## Predict the Threshold Sweep

Two scenarios use the same model scores:

- `miss_dominant`: false negative cost `20`, false positive cost `1`.
- `block_dominant`: false negative cost `5`, false positive cost `4`.

Predict which scenario selects the lower threshold. The synthetic costs simplify delayed and uneven real consequences.

In [ ]:
threshold_prediction = {"lower_threshold_scenario": "", "reason": "", "what_model_output_stays_fixed": ""}
assert all(value.strip() for value in threshold_prediction.values())

COST_SCENARIOS = {
    "miss_dominant": {"false_negative": 20.0, "false_positive": 1.0},
    "block_dominant": {"false_negative": 5.0, "false_positive": 4.0},
}
thresholds = np.unique(np.r_[np.linspace(0.02, 0.98, 97), DEFAULT_THRESHOLD])
assert DEFAULT_THRESHOLD in thresholds

## Focused TODO: Synchronized Sweep

For every threshold, call `metric_row` and add each scenario cost: `FN * false_negative + FP * false_positive`. Keep all rows aligned so one selected threshold drives the table and plots.

In [ ]:
def threshold_sweep(y_true, probabilities, thresholds, cost_scenarios):
    # TODO: return one dictionary per threshold with metrics and one cost per scenario.
    raise NotImplementedError("TODO: build the synchronized threshold sweep")

sweep = threshold_sweep(y_val, fraud_probability, thresholds, COST_SCENARIOS)
assert len(sweep) == len(thresholds)
assert any(np.isclose(row["threshold"], DEFAULT_THRESHOLD) for row in sweep)

In [ ]:
optimal_rows = {
    scenario: min(sweep, key=lambda row: row[f"cost_{scenario}"])
    for scenario in COST_SCENARIOS
}
for scenario, row in optimal_rows.items():
    print(scenario, {key: round(value, 4) if isinstance(value, float) else value for key, value in row.items()})
assert optimal_rows["miss_dominant"]["threshold"] != optimal_rows["block_dominant"]["threshold"]

## Inspect: One Threshold, Multiple Consequences

Choose one scenario below. The histogram, confusion matrix, metric traces, and cost trace use the same selected row. Observe what moves with the threshold and what remains a property of the fixed probability model.

In [ ]:
SELECTED_SCENARIO = "miss_dominant"
selected = optimal_rows[SELECTED_SCENARIO]
selected_threshold = selected["threshold"]
selected_predictions = (fraud_probability >= selected_threshold).astype(int)
selected_confusion = confusion_matrix(y_val, selected_predictions, labels=[0, 1])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].hist(fraud_probability[y_val == 0], bins=25, alpha=0.65, label="legitimate")
axes[0, 0].hist(fraud_probability[y_val == 1], bins=25, alpha=0.65, label="fraud")
axes[0, 0].axvline(selected_threshold, color="black", linestyle="--", label=f"t={selected_threshold:.2f}")
axes[0, 0].set(title="Fixed probabilities and selected threshold", xlabel="fraud probability", ylabel="examples")
axes[0, 0].legend()
image = axes[0, 1].imshow(selected_confusion, cmap="Greys")
for (row, col), value in np.ndenumerate(selected_confusion): axes[0, 1].text(col, row, str(value), ha="center", va="center")
axes[0, 1].set(title="Confusion matrix", xlabel="predicted", ylabel="actual", xticks=[0,1], yticks=[0,1])
axes[1, 0].plot(thresholds, [row["precision"] for row in sweep], label="precision")
axes[1, 0].plot(thresholds, [row["recall"] for row in sweep], label="recall")
axes[1, 0].plot(thresholds, [row["f1"] for row in sweep], label="F1")
axes[1, 0].axvline(selected_threshold, color="black", linestyle="--")
axes[1, 0].set(title="Metrics move with threshold", xlabel="threshold", ylabel="metric", ylim=(0,1.02)); axes[1,0].legend()
for scenario in COST_SCENARIOS: axes[1, 1].plot(thresholds, [row[f"cost_{scenario}"] for row in sweep], label=scenario)
axes[1, 1].axvline(selected_threshold, color="black", linestyle="--")
axes[1, 1].set(title="Scenario costs", xlabel="threshold", ylabel="synthetic cost"); axes[1,1].legend()
plt.tight_layout(); plt.show()

In [ ]:
pr_precision, pr_recall, _ = precision_recall_curve(y_val, fraud_probability)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(pr_recall, pr_precision, color="#2a6f97", label=f"PR curve; AP={default_metrics['average_precision']:.3f}")
ax.scatter(selected["recall"], selected["precision"], color="#c44536", s=80, label=f"selected t={selected_threshold:.2f}")
ax.set(title="Precision-recall overview and operating point", xlabel="recall", ylabel="precision", xlim=(0,1.02), ylim=(0,1.02)); ax.legend()
plt.show()
print(f"ROC AUC overview: {default_metrics['roc_auc']:.3f}; it ranks scores but does not choose the threshold.")

## Interpret and Defend

Cite confusion counts and consequence assumptions. A defensible answer may choose a different threshold from the cost minimum if it identifies an omitted constraint.

In [ ]:
threshold_decision = {
    "scenario": "",
    "chosen_threshold": "",
    "confusion_evidence": "",
    "metric_set": "",
    "cost_assumption": "",
    "what_auc_does_not_decide": "",
    "remaining_risk": "",
}
assert all(value.strip() for value in threshold_decision.values())

## Deliberate Failure and Recovery

The failed rule is `max accuracy => best model`. Compare it with a consequence-aware rule. Explain why the repair changes evaluation rather than changing the already fitted probability scores.

In [ ]:
accuracy_only_choice = max([("majority", majority_metrics), ("probability@0.5", default_metrics)], key=lambda item: item[1]["accuracy"])[0]
consequence_aware_choice = f"probability@{optimal_rows['miss_dominant']['threshold']:.2f}"
print("Accuracy-only choice:", accuracy_only_choice)
print("Consequence-aware operating point:", consequence_aware_choice)
assert majority_metrics["recall"] == 0.0

## Challenge: Change the Consequences, Not the Model

Create one additional bounded cost scenario, predict its threshold direction, recompute only the cost column, and explain the new operating point. Do not refit the classifier.

In [ ]:
RUN_OPTIONAL_SCENARIO = False
optional_scenario = {"false_negative": 12.0, "false_positive": 2.0}
optional_interpretation = {"prediction": "", "observed_threshold": "", "explanation": ""}
if RUN_OPTIONAL_SCENARIO:
    optional_rows = threshold_sweep(y_val, fraud_probability, thresholds, {"optional": optional_scenario})
    optional_best = min(optional_rows, key=lambda row: row["cost_optional"])
    print("Optional best row:", optional_best)
    assert all(value.strip() for value in optional_interpretation.values())

## Reflection, Takeaways, and Troubleshooting

1. High accuracy can coexist with zero rare-class recall.
2. Threshold movement changes class decisions, not the underlying probability ranking.
3. Precision, recall, F1, PR, and ROC/AUC answer different questions; none supplies stakeholder costs.
4. State the positive class and confusion orientation every time.

| Symptom | Likely cause | Recovery |
|---|---|---|
| Undefined metric warning | No positive predictions and unsafe defaults | Use `zero_division=0` and inspect counts |
| Costs appear reversed | FN/FP orientation swapped | Confirm `[[TN,FP],[FN,TP]]` |
| Threshold `0.5` missing | Sweep grid omitted it | Include it explicitly |
| Runtime exceeds 30 seconds | Dataset/model budget changed | Restore 6,000 examples and one logistic fit |

In [ ]:
assert majority_metrics["recall"] == 0.0
assert 0.55 <= default_metrics["recall"] <= 0.85
assert optimal_rows["miss_dominant"]["threshold"] != optimal_rows["block_dominant"]["threshold"]
assert all(value.strip() for value in threshold_decision.values())
print(f"LAB-D4-01 checkpoint passed in {time.perf_counter() - started:.2f}s: baseline exposed, thresholds synchronized, and a consequence-aware decision recorded.")

## Continue

Use the Day 4 guide debrief: [LAB-D4-01 Debrief - What Changed, and What Did Not?](../student-guide/day-4-student-guide.md#lab-d4-01-debrief---what-changed-and-what-did-not).